# Movie Recommendation System Using Collaborative Filtering

This project recommends movies based on patterns in MovieLens user ratings. It implements item-based collaborative filtering with mean-centered ratings, a sparse movie–user matrix, and cosine similarity.


## Dataset and approach

The MovieLens Latest Small dataset contains 100,836 ratings from 610 users for 9,742 movies. The recommender uses **ratings and movie metadata only**; tags and external links are not needed. User-mean centering reduces individual differences in rating scales. A minimum of five ratings per candidate helps limit unstable recommendations. Cosine similarity describes rating-pattern alignment, not predicted enjoyment.


## Imports

The implementation uses pandas and NumPy for data preparation, SciPy for the sparse matrix, and scikit-learn for cosine similarity.


In [ ]:
# Import the libraries used in the recommender system
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Display options make notebook output easier to read
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Load the MovieLens datasets

Place the two Excel files in the `data/` directory beside this notebook.


In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
required_files = [DATA_DIR / "movies.xlsx", DATA_DIR / "ratings.xlsx"]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing dataset files: {', '.join(missing)}")

# The movies workbook includes a title row before the column headers.
movies = pd.read_excel(DATA_DIR / "movies.xlsx", header=1)
ratings = pd.read_excel(DATA_DIR / "ratings.xlsx")
print("Movies:", movies.shape, "Ratings:", ratings.shape)


## Inspect the data

The analysis requires only `movies.xlsx` and `ratings.xlsx`.


In [ ]:
# Preview the movie information
display(movies.head())

# Preview the user ratings
display(ratings.head())

In [ ]:
# Basic quality checks
print("Unique users:", ratings["userId"].nunique())
print("Unique rated movies:", ratings["movieId"].nunique())
print("Total ratings:", len(ratings))
print("Missing values in movies.xlsx:", int(movies.isna().sum().sum()))
print("Missing values in ratings.xlsx:", int(ratings.isna().sum().sum()))
print("Duplicate rating rows:", int(ratings.duplicated().sum()))
print("Minimum rating:", ratings["rating"].min())
print("Maximum rating:", ratings["rating"].max())

## Mean-center ratings

Subtract each user’s average rating from their individual ratings to emphasize relative preferences. For example, a 5-star rating from a user whose mean is 3 stars becomes +2.


In [ ]:
# Calculate each user's average rating
ratings_prepared = ratings.copy()
ratings_prepared["user_mean"] = ratings_prepared.groupby("userId")["rating"].transform("mean")

# Mean-center every rating to reduce individual rating-scale bias
ratings_prepared["adjusted_rating"] = (
    ratings_prepared["rating"] - ratings_prepared["user_mean"]
)

display(ratings_prepared.head())

## Build the sparse movie–user matrix

Each row represents a movie and each column represents a user. Observed, mean-centered ratings are stored in a SciPy CSR matrix; unrated movie–user pairs are represented as zero.


In [ ]:
# Create stable row and column mappings for movie IDs and user IDs
rated_movie_ids = np.sort(ratings_prepared["movieId"].unique())
user_ids = np.sort(ratings_prepared["userId"].unique())

movie_to_row = {movie_id: i for i, movie_id in enumerate(rated_movie_ids)}
user_to_col = {user_id: i for i, user_id in enumerate(user_ids)}

row_indices = ratings_prepared["movieId"].map(movie_to_row).to_numpy()
col_indices = ratings_prepared["userId"].map(user_to_col).to_numpy()
values = ratings_prepared["adjusted_rating"].to_numpy()

movie_user_matrix = csr_matrix(
    (values, (row_indices, col_indices)),
    shape=(len(rated_movie_ids), len(user_ids))
)

print("Movie-user matrix shape:", movie_user_matrix.shape)
print("Stored non-zero values:", movie_user_matrix.nnz)

## Movie statistics

Rating counts and average ratings provide context for the ranked recommendations.


In [ ]:
# Precompute lookup tables used by the recommendation function
movie_title_lookup = movies.set_index("movieId")["title"].to_dict()
movie_genre_lookup = movies.set_index("movieId")["genres"].to_dict()
movie_rating_count = ratings.groupby("movieId").size()
movie_average_rating = ratings.groupby("movieId")["rating"].mean()

## 8. Recommendation Functions
The first helper searches for movies by title. The main `recommend_movies()` function then:

1. Finds the requested movie.
2. Gets the mean-centered user-rating vector.
3. Compares the vector with every rated movie using cosine similarity.
4. Removes the selected movie itself.
5. Filters out candidates with fewer than five ratings.
6. Sorts the remaining movies from most to least similar.
7. Returns the top ten recommendations.

In [ ]:
def find_movie_titles(search_text, max_results=15):
    """Return MovieLens titles containing the user's search text."""
    search_text = str(search_text).strip()
    matches = movies[
        movies["title"].str.contains(search_text, case=False, regex=False, na=False)
    ][["movieId", "title", "genres"]]
    return matches.head(max_results).reset_index(drop=True)


def recommend_movies(movie_title, n_recommendations=10, min_ratings=5):
    """
    Recommend movies with rating patterns similar to the entered movie.

    Parameters
    ----------
    movie_title : str
        Full or partial title of a movie in movies.xlsx.
    n_recommendations : int
        Number of recommendations to return. Default is 10.
    min_ratings : int
        Minimum number of ratings required for a candidate movie.

    Returns
    -------
    pandas.DataFrame
        Ranked recommendations with similarity, rating count, average rating,
        and genres.
    """
    movie_title = str(movie_title).strip()

    # First search for all titles containing the text entered by the user.
    matches = movies[
        movies["title"].str.contains(movie_title, case=False, regex=False, na=False)
    ].copy()

    if matches.empty:
        print(f'No movie containing "{movie_title}" was found in the dataset.')
        print("Try another title or use find_movie_titles('part of title').")
        return pd.DataFrame()

    # Prefer an exact title match when one exists.
    exact = matches[matches["title"].str.lower() == movie_title.lower()]
    if len(exact) == 1:
        selected = exact.iloc[0]
    elif len(matches) == 1:
        selected = matches.iloc[0]
    else:
        print("Multiple movies matched that search. Please enter a more specific title:")
        display(matches[["title", "genres"]].head(15).reset_index(drop=True))
        return pd.DataFrame()

    selected_movie_id = int(selected["movieId"])

    # Some titles may appear in movies.xlsx without a usable rating vector.
    if selected_movie_id not in movie_to_row:
        print(f'{selected["title"]} does not have enough rating data for this model.')
        return pd.DataFrame()

    # Calculate cosine similarity between the selected movie and all rated movies.
    selected_vector = movie_user_matrix[movie_to_row[selected_movie_id]]
    similarities = cosine_similarity(selected_vector, movie_user_matrix).ravel()

    # Build a result table with useful descriptive information.
    results = pd.DataFrame({
        "movieId": rated_movie_ids,
        "similarity": similarities
    })
    results["title"] = results["movieId"].map(movie_title_lookup)
    results["genres"] = results["movieId"].map(movie_genre_lookup)
    results["rating_count"] = results["movieId"].map(movie_rating_count).fillna(0).astype(int)
    results["average_rating"] = results["movieId"].map(movie_average_rating)

    # Exclude the selected movie and weakly supported/negative matches.
    results = results[
        (results["movieId"] != selected_movie_id) &
        (results["rating_count"] >= min_ratings) &
        (results["similarity"] > 0)
    ]

    # Rank primarily by cosine similarity, using rating count only as a tie-breaker.
    results = results.sort_values(
        ["similarity", "rating_count"], ascending=[False, False]
    ).head(n_recommendations).copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity"] = results["similarity"].round(3)
    results["average_rating"] = results["average_rating"].round(2)

    print(f'Recommendations based on: {selected["title"]}')
    return results[[
        "rank", "title", "genres", "similarity", "rating_count", "average_rating"
    ]].reset_index(drop=True)

## 9. User Input and Demonstration

In [ ]:
# USER INPUT CELL
# Replace the title below with any movie title from the MovieLens dataset.
favorite_movie = "Toy Story (1995)"

# Optional console prompt instead of the line above:
# favorite_movie = input("Enter a movie you like: ")

recommendations = recommend_movies(favorite_movie, n_recommendations=10)
display(recommendations)

### Example title search
If the exact MovieLens title is unknown, the helper below can be used to find matching titles before running the recommender.

In [ ]:
# Example: search for titles containing the words "Dark Knight"
display(find_movie_titles("Dark Knight"))

## Interpreting recommendations

The example uses *Toy Story (1995)*. Recommended titles are ranked by similarity of user-rating patterns; genre labels are displayed for context but are not used to compute similarity. Results are not predicted star ratings and do not guarantee that a viewer will enjoy a recommended film.


## Limitations and future improvements

- Sparse ratings limit information about less-popular movies and create a cold-start problem.
- A single favorite movie captures less preference information than a user's rating history.
- The method does not incorporate genres, tags, cast, or directors into the similarity calculation.
- The MovieLens Latest Small dataset is a research dataset, not a live catalog.
- Future work could evaluate recommendation quality on held-out interactions, incorporate content features, or use matrix factorization.


## Conclusion

This notebook demonstrates an interpretable item-based collaborative filtering pipeline: load and inspect ratings, mean-center by user, construct a sparse movie–user matrix, compute cosine similarity, filter weakly supported candidates, and return ranked movie recommendations.


## References

Harper, F. M., & Konstan, J. A. (2015). *The MovieLens Datasets: History and Context*. ACM Transactions on Interactive Intelligent Systems, 5(4), Article 19. https://doi.org/10.1145/2827872

GroupLens Research. (2018). *MovieLens Latest Small Dataset / README*. https://grouplens.org/datasets/movielens/latest/

Sarwar, B., Karypis, G., Konstan, J. A., & Riedl, J. (2001). *Item-based collaborative filtering recommendation algorithms*. Proceedings of the 10th International Conference on World Wide Web, 285-295. https://doi.org/10.1145/371920.372071

Scikit-learn Developers. *cosine_similarity - scikit-learn documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html